# S5 — physics-consistency scoring of synthetic mammograms + failure investigation (no training; TEST SET NOT USED)
Model: the epoch-40 clean baseline (seed 42). Every per-image score has a FIXED direction (lower = better), fixed before the run.

| part | what |
|---|---|
| pool | 300 generated images / class (guidance 2, DDIM-50, the usual noise bank) |
| F1 | **physics**: score Fokker–Planck residual of the model at the noised image (t = 10, 50), validated FD estimator |
| F2 | **physics**: film/digitiser noise (Poisson–Gaussian noise-level function fitted on real train-fit images) |
| F3 | model's own denoising error (t = 50, 200, 500), normalised by real validation images |
| F4 | trivial: brightness / saturation / HF energy / breast area distance to real |
| F5 | reference only: nearest-real distance in the KID feature space (never a pass criterion) |
| T1 | reliability of F1 (independent re-scoring of 80 images; Spearman ≥ 0.5) |
| T2 | filter: keep best 50 % per class → KID-mammo ≤ 0.90× median of 1000 random 50 % subsets and below their 2.5th percentile |
| T3 / E2 | real vs generated detector: AUC ≥ 0.65 with CI > 0.5 (per t for F1) |
| E3 | KID of checkpoints 35 / 40 / 41 / 42 / 44 and of their weight average (descriptive: checkpoint-selection variability) |
| E4 | Epsilon Scaling at sampling, s = 1.005 / 1.01 / 1.02 (helps if KID ≤ 0.90× with paired CI < 0) |

**How to run (account with the S3 and S4a notebooks)**
1. Add Input → **clean-data-baseline-retraining**, **S2 baseline audit**, **S3** output and **S4a** output.
2. Settings → Accelerator **GPU T4 ×2** · Internet **ON**.
3. **Save Version → Save & Run All (Commit)**, then stop the interactive session.

**Time:** setup ≈ 10 min + smoke test ≈ 10 min + ≈ 3.5 h of the two GPU queues + analysis ≈ 10 min → **≈ 4 h of GPU quota**
(hard limit `BUDGET_H = 9.0` h). **Send back:** `S5/S5_report.txt` + `S5/grid_*.png` (or `S5_results.zip`).

In [ ]:
# ===================== CELL 1 — CONFIG =====================
BUDGET_H = 9.0            # hard limit for this session; the run needs ~4 h
POOL_PER_CLASS = 300
E3_PER_CLASS = 100        # same size as the S3 / S4 evaluation sets (their epoch-42 / 44 samples are reused)
EPS_SCALES = [1.005, 1.01, 1.02]
RUN_SMOKE = True
import os, time
os.environ["PA_OUT"] = "/kaggle/working/S5"
os.makedirs(os.environ["PA_OUT"], exist_ok=True)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
print("S5 physics scoring | budget", BUDGET_H, "h | output", os.environ["PA_OUT"])

In [ ]:
# ===================== CELL 2 — setup (needs Internet ON) =====================
import os, re, sys, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
    subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
if not os.path.exists("/tmp/pip_done"):
    # Kaggle runs Python 3.13: install the repo requirements once, never force-reinstall numpy / pandas
    subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                   "pip install -q -r /tmp/req.txt", shell=True, check=True)
    subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
    open("/tmp/pip_done", "w").write("ok")
probe = ("import sys, numpy, pandas, scipy, torch, torchmetrics, monai, generative, PIL; "
         "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
         "'| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, '| monai', monai.__version__)")
env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
if env_chk.returncode != 0:
    print(env_chk.stderr[-2500:])
    raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
print("environment OK:", env_chk.stdout.strip())
p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
s = open(p).read()
s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers is not installed
if s2 != s:
    open(p, "w").write(s2)
print("flash attention ->", "patched off" if s2 != s else "already off")
m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader (used later in A6): hard-coded config path
s = open(m).read()
s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
if s2 != s:
    open(m, "w").write(s2)
print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — code + clean data (reused) + input checks + job runner =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.chdir(WD)
FILES = {"phaseA_common.py": r'''# phaseA_common.py - shared helpers for the Phase A diagnostics (no training happens in Phase A).
# Model, schedule, data domain and conditioning are taken from the Experiment-2 code (train_arm2.py / eval_arm.py):
#   * model: model_architectures.MonaiDDPM().unet (MONAI-Generative DiffusionModelUNet, flash attention off)
#   * schedule: MonaiDDPM().scheduler.alphas_cumprod (linear beta 1e-4..0.02, T=1000)
#   * images: per-image min-max to [-1,1]; physics statistic on (x+1)/2
#   * conditioning: context = label as float (B,1,1); -1 = null label (classifier-free guidance)
import os, sys, glob, json, math, time
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
if WD not in sys.path:
    sys.path.insert(0, WD)
OUT = os.environ.get("PA_OUT", "/kaggle/working/phaseA")
os.makedirs(OUT, exist_ok=True)


def log(tag, *m):
    print(f"[{tag}]", *m, flush=True)


_INDEX = None


def find(name, roots=("/kaggle/input", "/kaggle/working")):
    """all files with this exact name in the attached inputs (one directory walk per process; image folders skipped)"""
    global _INDEX
    if _INDEX is None:
        _INDEX = {}
        for r in roots:
            for dp, dns, fns in os.walk(r):
                dns[:] = [d for d in dns if d not in ("jpeg", "U24_synthesis", ".git", "phaseA")]
                for fn in fns:
                    _INDEX.setdefault(fn, []).append(os.path.join(dp, fn))
    return sorted(set(_INDEX.get(name, [])))


def main_ckpt():
    """full-data Experiment-2 baseline, epoch 40 (the main Phase A checkpoint)"""
    h = find("base2_w_ep040.pt")
    if not h:
        raise SystemExit("base2_w_ep040.pt not found: Add Input -> Your Work -> the Experiment 2 notebook "
                         "(notebook64a7f79592), then run again")
    return h[0]


def clean_ckpt(seed, epoch=40):
    """clean-data baseline (step 2 of the timeline): base_clean_s<seed>_w_ep<epoch>.pt"""
    h = find(f"base_clean_s{seed}_w_ep{epoch:03d}.pt")
    if not h:
        raise SystemExit(f"base_clean_s{seed}_w_ep{epoch:03d}.pt not found: Add Input -> Your Work -> the clean-baseline "
                         "notebook (its latest output version), then run again")
    return h[0]


def optional_25_ckpt():
    """any 25 %-data baseline at epoch 45 from Experiment 3 or 4 (optional)"""
    for s in (42, 43, 44, 45, 46, 47, 48):
        h = find(f"base3_s{s}_w_ep045.pt")
        if h:
            return h[0]
    return None


def schedule(dev):
    from model_architectures import MonaiDDPM
    m = MonaiDDPM(lr=1e-5, T=1000)
    ac = m.scheduler.alphas_cumprod.detach().double().to(dev)
    unet = m.unet
    del m
    return unet, ac


def load_model(path, dev, dtype=torch.float32):
    """returns (unet in eval mode, alphas_cumprod float64, snapshot meta)"""
    snap = torch.load(path, map_location="cpu", weights_only=False)
    unet, ac = schedule(dev)
    sd = snap["unet"] if "unet" in snap else snap["models"]["net"]
    unet.load_state_dict({k: v.float() for k, v in sd.items()})
    unet = unet.to(dev).to(dtype).eval()
    for p in unet.parameters():
        p.requires_grad_(False)
    meta = {k: snap[k] for k in ("epoch", "step", "lam", "tau") if k in snap}
    meta["nlf"] = snap.get("nlf")
    meta["path"] = path
    del snap
    return unet, ac, meta


def load_cache(split):
    d = np.load(f"/tmp/{split}_cache.npz")
    return d["x"], d["y"]


def load_clean_meta(split):
    """clean cache WITH patient ids and md5 - same order as /tmp/<split>_cache.npz (written by prepare_clean.py)"""
    d = np.load(f"/kaggle/working/clean_data/{split}_cache.npz")
    return d["x"], d["y"], d["patient"].astype(str), d["md5"].astype(str)


def val_mask(patients, frac=0.15, seed=0):
    """patient-level VALIDATION split of the clean TRAIN set (classifiers, gates, guidance choice only).
    The diffusion model itself was trained on all of train. Fixed seed -> identical in every script/session."""
    ups = sorted(set(patients))
    rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])


def minmax_pm1(x):
    """x: torch (B,1,H,W) any range -> per-image min-max to [-1,1] (training domain)"""
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1


def to_model_domain(x_u8, dev):
    return minmax_pm1(torch.from_numpy(x_u8).float().unsqueeze(1).to(dev))


def ctx_of(labels, dev, dtype=torch.float32):
    """labels: list/array of 0, 1 or -1 (null) -> context tensor (B,1,1)"""
    return torch.as_tensor(np.asarray(labels, dtype=np.float32), device=dev).view(-1, 1, 1).to(dtype)


def eps_pred(unet, x, t_int, ctx, amp=False, chunk=8):
    """epsilon prediction. amp=True: exactly as in training (fp32 weights + fp16 autocast); amp=False: the weights'
    own dtype. Large batches are split into chunks (the non-flash attention at 64x64 needs ~0.5 GB per image)."""
    if chunk and x.shape[0] > chunk:
        if ctx.shape[0] == 1:
            ctx = ctx.expand(x.shape[0], *ctx.shape[1:])
        return torch.cat([eps_pred(unet, x[i:i + chunk], t_int, ctx[i:i + chunk], amp, None)
                          for i in range(0, x.shape[0], chunk)])
    tb = torch.full((x.shape[0],), int(t_int), device=x.device, dtype=torch.long)
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            out = unet(x, timesteps=tb, context=ctx)
        return out.float()
    return unet(x, timesteps=tb, context=ctx)


def save_json(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(obj, fh, indent=1, default=float)
''',
         "fp_core.py": r'''# fp_core.py - the Fokker-Planck (score-FPE) residual and its exact "manual recompute" gradient.
# Shared by the Phase A-2 diagnostics and (later) the Phase B/C training, so both use byte-identical maths.
# VP-SDE of the DDPM (tau = t/T, beta(tau) = -d log abar / d tau), score s = -eps_hat / sqrt(1 - abar_t):
#     d_tau s = 1/2 beta grad_x( x.s + div s + |s|^2 )
# projected on a unit direction u, Hutchinson probe v (Rademacher) for d_u(div s):
#     r(u,v) = u.d_tau s - 1/2 beta [ u.s + (J_s u).(x + 2 s) + v.(d_u J_s) v ]
# all derivatives by central finite differences (per-pixel step p; h_u = 256 p for unit u on 256x256, h_v = p);
# d_tau s by +-k integer steps. Product of two independent probes = unbiased estimate of (u.R)^2.
# Verified in Phase A: unit test vs autograd (1e-13), analytic Gaussian score -> residual ~0, wrong score -> O(1).
import math
import torch

T_STEPS = 1000
DTAU = 1.0 / T_STEPS


def p_of_t(t):
    """finite-difference step per t (fixed before Phase B from the A4 accuracy table: <= 1 % error vs the exact JVP)"""
    return 1e-3 if t <= 20 else (3e-3 if t <= 50 else 1e-2)


def beta_eff(logab, t):
    return float(-(logab[t + 1] - logab[t - 1]) / (2 * DTAU))


def plan(x, t, u, v1, v2, p, k=1):
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in ((1, v1), (2, v2)):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals(s, x, u, v1, v2, p, k, beta):
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    base = (u * s["s0"]).sum() + (Ju * (x + 2 * s["s0"])).sum()
    out = []
    for j, v in ((1, v1), (2, v2)):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        out.append(A - 0.5 * beta * (base + D2))
    return out[0], out[1], A


def coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w=1.0):
    """dL/ds_k for L = w r1 r2 (shared evaluations: w (r1 + r2) dr/ds_k; probe-specific: w r_other dr_j/ds_k)"""
    hu, hv = 256.0 * p, p
    b = 0.5 * beta
    Ju = (s["up"] - s["um"]) / (2 * hu)
    shared = {"tp": u / (2 * k * DTAU), "tm": -u / (2 * k * DTAU), "s0": -b * (u + 2 * Ju),
              "up": -b * (x + 2 * s["s0"]) / (2 * hu), "um": b * (x + 2 * s["s0"]) / (2 * hu)}
    g = {key: w * (r1 + r2) * val for key, val in shared.items()}
    for j, v, other in ((1, v1, r2), (2, v2, r1)):
        c = b * v / (4 * hu * hv)
        g[f"pp{j}"], g[f"pm{j}"], g[f"mp{j}"], g[f"mm{j}"] = -w * other * c, w * other * c, w * other * c, -w * other * c
    return g


def evaluate_nograd(sfn, E):
    s = {}
    with torch.no_grad():
        for tt in sorted({e[2] for e in E}):
            keys = [e[0] for e in E if e[2] == tt]
            out = sfn(torch.cat([e[1] for e in E if e[2] == tt]), tt)
            for i, kk in enumerate(keys):
                s[kk] = out[i:i + 1]
    return s


def fp_loss_and_grad(sfn, x, t, u, v1, v2, p, k, beta, w=1.0, group=1, backward=True):
    """exact gradient of L = w r1 r2 accumulated into .grad (memory of ONE network evaluation). Returns (L, r1, r2, lhs)."""
    E = plan(x, t, u, v1, v2, p, k)
    s = evaluate_nograd(sfn, E)
    with torch.no_grad():
        r1, r2, A = residuals(s, x, u, v1, v2, p, k, beta)
        g = coeffs(s, x, u, v1, v2, p, k, beta, r1, r2, w)
    if backward:
        for tt in sorted({e[2] for e in E}):
            grp = [e for e in E if e[2] == tt]
            for i in range(0, len(grp), group):
                part = grp[i:i + group]
                out = sfn(torch.cat([e[1] for e in part]), tt)
                (torch.cat([g[e[0]] for e in part]) * out).sum().backward()
    return float(w * r1 * r2), float(r1), float(r2), float(A)


def plan_n(x, t, u, vs, p, k=1):
    """same as plan() but for any number of probes vs (list of tensors)"""
    hu, hv = 256.0 * p, p
    E = [("tp", x, t + k), ("tm", x, t - k), ("s0", x, t), ("up", x + hu * u, t), ("um", x - hu * u, t)]
    for j, v in enumerate(vs):
        E += [(f"pp{j}", x + hu * u + hv * v, t), (f"pm{j}", x + hu * u - hv * v, t),
              (f"mp{j}", x - hu * u + hv * v, t), (f"mm{j}", x - hu * u - hv * v, t)]
    return E


def residuals_n(s, x, u, vs, p, k, beta):
    """r(u, v_j) for every probe j (same formula as residuals()); returns (list of r_j, lhs, terms dict)"""
    hu, hv = 256.0 * p, p
    A = (u * (s["tp"] - s["tm"])).sum() / (2 * k * DTAU)
    Ju = (s["up"] - s["um"]) / (2 * hu)
    us, Jt = (u * s["s0"]).sum(), (Ju * (x + 2 * s["s0"])).sum()
    rs, d2s = [], []
    for j, v in enumerate(vs):
        D2 = (v * (s[f"pp{j}"] - s[f"pm{j}"] - s[f"mp{j}"] + s[f"mm{j}"])).sum() / (4 * hu * hv)
        rs.append(A - 0.5 * beta * (us + Jt + D2)); d2s.append(D2)
    return rs, A, {"us": 0.5 * beta * us, "Jt": 0.5 * beta * Jt, "D2": [0.5 * beta * d for d in d2s]}


def make_score_fn(unet, ac, ctx_value, amp=False, chunk=4):
    """score function s(x, t) of the network for one class label (context value); fp32 by default (A4: fp16 unusable).
    Batches larger than `chunk` are split (non-flash attention at 64x64 needs ~0.5 GB per image in fp32)."""
    def sfn(xb, tt):
        if xb.shape[0] > chunk:
            return torch.cat([sfn(xb[i:i + chunk], tt) for i in range(0, xb.shape[0], chunk)])
        c = torch.full((xb.shape[0], 1, 1), float(ctx_value), device=xb.device, dtype=xb.dtype)
        tb = torch.full((xb.shape[0],), int(tt), device=xb.device, dtype=torch.long)
        if amp:
            with torch.autocast("cuda", dtype=torch.float16):
                e = unet(xb, timesteps=tb, context=c).float()
        else:
            e = unet(xb, timesteps=tb, context=c)
        return -e / math.sqrt(1.0 - float(ac[tt]))
    return sfn


def sref_interp(table, t):
    """log-linear interpolation of the reference scale table {t: RMS(u.d_tau s)} used to normalise the loss"""
    ts = sorted(int(k) for k in table)
    t = min(max(t, ts[0]), ts[-1])
    for a, b in zip(ts[:-1], ts[1:]):
        if a <= t <= b:
            la, lb = math.log(table[str(a)] if str(a) in table else table[a]), math.log(table[str(b)] if str(b) in table else table[b])
            return math.exp(la + (lb - la) * (t - a) / max(b - a, 1))
    return table[str(ts[-1])] if str(ts[-1]) in table else table[ts[-1]]
''',
         "a6_classifier.py": r'''# a6_classifier.py - A6: real-data benign/malignant classifier on the CLEAN split (+ shared training helpers).
# Used as (i) the "real-only" reference of the later downstream test, (ii) the judge of class conditioning (A7/A9),
# (iii) the "mammography feature" extractor for KID-mammo.
# PRE-STATED RECIPE (no tuning, no epoch selection): torchvision ConvNeXt-Tiny with ImageNet-1k weights,
# 256 px grayscale -> 3 channels, per-image min-max, ImageNet normalisation, AdamW lr 1e-4 wd 0.05, batch 32,
# 15 epochs, cosine schedule with 1 warm-up epoch, horizontal flip + small affine, class-balanced cross-entropy,
# fp16 autocast. FINAL epoch is used. Seeds 0, 1, 2. Trained on clean TRAIN minus the patient-level VALIDATION split.
# PRE-STATED RULE (validation only - the test set is never used for a decision):
#   usable as judge if the mean VALIDATION AUC over the 3 seeds is >= 0.65.  Test AUC is reported, not used.
import os, sys, time, json, math, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_cache, load_clean_meta, val_mask, OUT

TAG = "A6"
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)


def prep(x01):
    """(B,1,H,W) any range -> per-image min-max -> 3ch ImageNet-normalised"""
    mn = x01.amin(dim=(1, 2, 3), keepdim=True); mx = x01.amax(dim=(1, 2, 3), keepdim=True)
    x = (x01 - mn) / (mx - mn + 1e-8)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD


def build():
    import torchvision
    m = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1")
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m


def auc(scores, labels):
    """Mann-Whitney AUC with tie handling (no sklearn dependency)"""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    order = np.argsort(s); r = np.empty(len(s)); r[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):
        m = s == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


@torch.no_grad()
def predict(m, X_u8, bs=64):
    m.eval(); out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.as_tensor(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(m(prep(xb)).float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def augment(x, g):
    B = x.shape[0]
    u = lambda: torch.rand(B, generator=g, device=dev) * 2 - 1
    sx, sy, shx, shy = 1 + 0.05 * u(), 1 + 0.05 * u(), 0.05 * u(), 0.05 * u()
    z = torch.zeros(B, device=dev)
    th = torch.stack([torch.stack([sx, shx, z], 1), torch.stack([shy, sy, z], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(th, list(x.shape), align_corners=False), padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=g, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)


def train_clf(X_u8, y, seed, epochs=15, bs=32, tag=TAG):
    """the fixed recipe above; returns the model after the FINAL epoch"""
    t0 = time.time()
    Xg, Yg = torch.from_numpy(X_u8).to(dev), torch.from_numpy(np.asarray(y)).long().to(dev)
    w = torch.tensor([len(y) / (2 * max(1, (np.asarray(y) == c).sum())) for c in (0, 1)], device=dev, dtype=torch.float32)
    torch.manual_seed(seed)
    m = build().to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=0.05)
    spe = math.ceil(len(y) / bs); total = spe * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / spe) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    g = torch.Generator(device=dev).manual_seed(seed)
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(seed * 100 + ep)).to(dev)
        tl = 0.0
        for i in range(spe):
            idx = perm[i * bs:(i + 1) * bs]
            xb = augment(Xg[idx].float().div(255).unsqueeze(1), g)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(prep(xb)).float(), Yg[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl += float(loss)
        if (ep + 1) % 5 == 0 or ep == 0:
            log(tag, f"seed {seed} epoch {ep + 1}/{epochs} train loss {tl / spe:.4f} | {(time.time() - t0) / 60:.1f} min")
    del Xg, Yg
    return m


def boot_auc(p, y, n=1000, seed=0):
    r = np.random.default_rng(seed); vals = []
    i0, i1 = np.where(np.asarray(y) == 0)[0], np.where(np.asarray(y) == 1)[0]
    for _ in range(n):
        b = np.concatenate([r.choice(i0, len(i0)), r.choice(i1, len(i1))])
        vals.append(auc(p[b], np.asarray(y)[b]))
    return [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))]


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", default="0,1,2")
    ap.add_argument("--epochs", type=int, default=15)
    a = ap.parse_args()
    x, y, pat, _ = load_clean_meta("train"); xte, yte = load_cache("test")
    vm = val_mask(pat)
    res = {"recipe": "convnext_tiny IMAGENET1K_V1, AdamW 1e-4 wd .05, bs 32, 15 ep cosine (1 warm-up), flip+affine, "
                     "balanced CE, final epoch; trained on train minus validation patients",
           "n_train_fit": int((~vm).sum()), "n_val": int(vm.sum()), "n_val_patients": len(set(pat[vm])),
           "n_test": int(len(yte)), "seeds": {}}
    log(TAG, f"fit on {res['n_train_fit']} images | validation {res['n_val']} images ({res['n_val_patients']} patients)")
    pv_sum = np.zeros(int(vm.sum()))
    for seed in [int(s) for s in a.seeds.split(",")]:
        t0 = time.time()
        m = train_clf(x[~vm], y[~vm], seed, a.epochs)
        pv, pt = predict(m, x[vm]), predict(m, xte)
        pv_sum += pv
        res["seeds"][str(seed)] = {"val_auc": auc(pv, y[vm]), "test_auc_reported_only": auc(pt, yte),
                                   "val_balanced_acc": float(np.mean([((pv >= 0.5) == (y[vm] == 1))[y[vm] == c].mean() for c in (0, 1)])),
                                   "minutes": (time.time() - t0) / 60}
        torch.save({k: v.half() for k, v in m.state_dict().items()}, os.path.join(OUT, f"a6_clf_s{seed}.pt"))
        log(TAG, f"seed {seed}: VALIDATION AUC {res['seeds'][str(seed)]['val_auc']:.3f} | test AUC (reported only) "
                 f"{res['seeds'][str(seed)]['test_auc_reported_only']:.3f} | {(time.time() - t0) / 60:.1f} min")
        del m; torch.cuda.empty_cache()
    va = [v["val_auc"] for v in res["seeds"].values()]
    res["mean_val_auc"] = float(np.mean(va)); res["sd_val_auc"] = float(np.std(va, ddof=1)) if len(va) > 1 else 0.0
    pe = pv_sum / len(va)
    res["ensemble_val_auc"] = auc(pe, y[vm]); res["ensemble_val_auc_ci95"] = boot_auc(pe, y[vm])
    res["mean_test_auc_reported_only"] = float(np.mean([v["test_auc_reported_only"] for v in res["seeds"].values()]))
    res["usable_as_judge"] = bool(res["mean_val_auc"] >= 0.65)
    save_json(res, "A6_classifier.json")
    log(TAG, f"mean VALIDATION AUC {res['mean_val_auc']:.3f} +- {res['sd_val_auc']:.3f} (ensemble {res['ensemble_val_auc']:.3f}) "
             f"-> usable as judge: {res['usable_as_judge']}")
''',
         "pinn_physics.py": r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''',
         "s5_generate.py": r'''# s5_generate.py - S5 sample generation (no training). Same DDIM-50 + classifier-free guidance sampler and the SAME
# noise bank as s3_eval / s4_eval (noise seed 2025 + 100000*label + batch, batch 16, benign first), so image i of every
# set starts from the same noise and label (paired comparisons between checkpoints / sampler settings).
#   --ckpts a.pt[,b.pt,...]  one snapshot, or several -> their weights are averaged (uniform checkpoint averaging)
#   --eps_scale s            Epsilon Scaling (Ning et al., ICLR 2024): the guided noise prediction is divided by s at
#                            every step (s = 1: unchanged sampler)
import os, sys, time, json, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, schedule, OUT

ap = argparse.ArgumentParser()
ap.add_argument("--name", required=True)
ap.add_argument("--ckpts", required=True)
ap.add_argument("--n_per_class", type=int, default=100)
ap.add_argument("--gs", type=float, default=2.0)
ap.add_argument("--eps_scale", type=float, default=1.0)
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--noise_seed", type=int, default=2025)
a = ap.parse_args()
TAG = f"S5-gen-{a.name}"
dev = torch.device("cuda")
paths = a.ckpts.split(",")
unet, ac = schedule(dev)
avg = None
for p in paths:
    sd = torch.load(p, map_location="cpu", weights_only=False)
    sd = sd["unet"] if "unet" in sd else sd["models"]["net"]
    if avg is None:
        avg = {k: v.float().clone() for k, v in sd.items()}
    else:
        for k in avg:
            avg[k] += sd[k].float()
    del sd
for k in avg:
    avg[k] /= len(paths)
unet.load_state_dict(avg); del avg
unet = unet.to(dev).half().eval()
for q in unet.parameters():
    q.requires_grad_(False)
acf = ac.float(); T = acf.numel()
log(TAG, f"{len(paths)} checkpoint(s) {'(weights averaged)' if len(paths) > 1 else ''}: {paths} | {a.n_per_class}/class | gs {a.gs} | "
         f"eps_scale {a.eps_scale} | {a.steps} DDIM steps")

@torch.no_grad()
def ddim(label, n, gen):
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    cond = torch.full((n, 1, 1), float(label), device=dev)
    for i, t in enumerate(ts):
        ab_t = acf[t]
        ab_prev = acf[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
        ctx = torch.cat([-torch.ones_like(cond), cond]).half()
        eu, ec = unet(torch.cat([x, x]).half(), timesteps=tb, context=ctx).float().chunk(2)
        eps = (eu + a.gs * (ec - eu)) / a.eps_scale
        x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)

t0 = time.time(); imgs, labels = [], []
for label in (0, 1):
    done, b = 0, 0
    while done < a.n_per_class:
        k = min(16, a.n_per_class - done)
        out = ddim(label, k, torch.Generator(device=dev).manual_seed(a.noise_seed + 100000 * label + b))[:, 0].cpu()
        imgs.append((out.numpy() * 255).round().astype(np.uint8)); labels += [label] * k; done += k; b += 1
    log(TAG, f"label {label}: {done} images | {(time.time() - t0) / 60:.1f} min")
G = np.concatenate(imgs); L = np.array(labels, dtype=np.int64)
np.savez_compressed(os.path.join(OUT, f"gen_{a.name}.npz"), x=G, y=L)
from PIL import Image
kk = min(8, a.n_per_class)
Image.fromarray(np.concatenate([np.concatenate([G[j + c * a.n_per_class] for j in range(kk)], 1) for c in (0, 1)], 0)).save(
    os.path.join(OUT, f"grid_{a.name}.png"))
save_json({"name": a.name, "ckpts": paths, "n_per_class": a.n_per_class, "gs": a.gs, "eps_scale": a.eps_scale, "steps": a.steps,
           "minutes": (time.time() - t0) / 60}, f"gen_{a.name}.json")
log(TAG, f"saved gen_{a.name}.npz ({len(G)} images) in {(time.time() - t0) / 60:.1f} min")
''',
         "s5_score.py": r'''# s5_score.py - S5 per-image scores computed with the epoch-40 clean baseline (no training):
#   F1 (physics: score Fokker-Planck consistency) - each image x0 (generated or real) is noised to x_t at t in --t; the
#      projected score-FPE residual r(u,v) of the MODEL at x_t is measured with the validated fp32 finite-difference
#      estimator (fp_core, step table p(t)), one unit direction u and two Rademacher probes, with the image's class label:
#      P(t) = r1 r2 / S_ref(t)^2 (unbiased for (u.R)^2 / S_ref^2). Per-image draws come from a fixed seed per image.
#   F3 (non-physics: the model's own denoising error) - eps-MSE of the model on the image noised at t in --mse_t
#      (fixed noise per image, fp16 autocast exactly as in training, true label).
# Sets: real = the 347 validation images; pool = a generated set (gen_*.npz); retest = the first --n_retest pool images
# scored again with an independent seed (test-retest reliability of F1).
import os, sys, time, json, math, argparse
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, minmax_pm1, load_model, find, eps_pred, ctx_of, OUT
import fp_core as fc

ap = argparse.ArgumentParser()
ap.add_argument("--set", choices=["real", "pool", "retest"], required=True)
ap.add_argument("--gen", default="")
ap.add_argument("--ckpt", required=True)
ap.add_argument("--t", default="10,50")
ap.add_argument("--mse_t", default="50,200,500")
ap.add_argument("--n_retest", type=int, default=80)
ap.add_argument("--seed", type=int, default=5)
ap.add_argument("--limit", type=int, default=0)
a = ap.parse_args()
TAG = f"S5-score-{a.set}"
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dev = torch.device("cuda")
TT = [int(v) for v in a.t.split(",")]; MT = [int(v) for v in a.mse_t.split(",")]
SREF = json.load(open(find("A8_fp_clean.json")[0]))["S_ref_table"]

if a.set == "real":
    xtr, ytr, pat, _ = load_clean_meta("train"); vm = val_mask(pat)
    X, Y = xtr[vm], ytr[vm]
else:
    d = np.load(a.gen); X, Y = d["x"], d["y"]
    if a.set == "retest":
        X, Y = X[:a.n_retest], Y[:a.n_retest]
if a.limit:
    X, Y = X[:a.limit], Y[:a.limit]
seed = a.seed + (1 if a.set == "retest" else 0)
unet, ac, meta = load_model(a.ckpt, dev)                    # fp32, eval, no grad
LOGAB = torch.log(ac.cpu())
SF = {lab: fc.make_score_fn(unet, ac, lab, chunk=12) for lab in (0, 1)}
n = len(Y); H_, W_ = X.shape[1:]
log(TAG, f"{n} images | FP t {TT} | denoising-error t {MT} | model {a.ckpt} (epoch {meta.get('epoch')}) | seed {seed}")

R = np.zeros((n, len(TT), 2)); A_ = np.zeros((n, len(TT)))
t0 = time.time()
for i in range(n):
    x0 = minmax_pm1(torch.from_numpy(X[i:i + 1]).float().unsqueeze(1)).to(dev)
    g = torch.Generator().manual_seed(seed * 1_000_003 + i)
    for ti, t in enumerate(TT):
        ab = float(ac[t]); p = fc.p_of_t(t)
        nz = torch.randn((1, 1, H_, W_), generator=g); u = torch.randn((1, 1, H_, W_), generator=g); u = u / u.norm()
        vs = [(torch.randint(0, 2, (1, 1, H_, W_), generator=g) * 2 - 1).float().to(dev) for _ in range(2)]
        x = math.sqrt(ab) * x0 + math.sqrt(1 - ab) * nz.to(dev); u = u.to(dev)
        s = fc.evaluate_nograd(SF[int(Y[i])], fc.plan_n(x, t, u, vs, p))
        rs, A, _ = fc.residuals_n(s, x, u, vs, p, 1, fc.beta_eff(LOGAB, t))
        R[i, ti] = [float(r) for r in rs]; A_[i, ti] = float(A)
    if (i + 1) % 50 == 0 or i == n - 1:
        log(TAG, f"  FP score {i + 1}/{n} | {(time.time() - t0) / 60:.1f} min")
sr = np.array([fc.sref_interp(SREF, t) for t in TT])
P = R[:, :, 0] * R[:, :, 1] / sr[None, :] ** 2

MSE = np.zeros((n, len(MT)))
gm = torch.Generator().manual_seed(seed * 7 + 3)
acf = ac.float()
with torch.no_grad():
    for i in range(0, n, 16):
        x0 = minmax_pm1(torch.from_numpy(X[i:i + 16]).float().unsqueeze(1)).to(dev); k = x0.shape[0]
        for ti, t in enumerate(MT):
            eps = torch.randn(x0.shape, generator=gm).to(dev); ab = acf[t]
            xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
            eh = eps_pred(unet, xt, t, ctx_of(Y[i:i + k], dev), amp=True, chunk=8)
            MSE[i:i + k, ti] = ((eh - eps) ** 2).mean((1, 2, 3)).cpu().numpy()
np.savez(os.path.join(OUT, f"score_{a.set}.npz"), R=R, A=A_, P=P, H=P.mean(1), MSE=MSE, y=Y, t=np.array(TT), mse_t=np.array(MT))
save_json({"set": a.set, "n": n, "t": TT, "mse_t": MT, "seed": seed, "minutes": (time.time() - t0) / 60,
           "H_mean": float(P.mean()), "P_mean_per_t": P.mean(0).tolist()}, f"score_{a.set}.json")
log(TAG, f"done: H mean {P.mean():.4f} | per t " + " ".join(f"{t}:{v:.3f}" for t, v in zip(TT, P.mean(0)))
         + f" | denoising MSE per t " + " ".join(f"{t}:{v:.5f}" for t, v in zip(MT, MSE.mean(0))) + f" | {(time.time() - t0) / 60:.1f} min")
''',
         "s5_stats.py": r'''# s5_stats.py - S5 statistics (pure numpy; called by s5_analyze.py, re-runnable on CPU from S5_data.npz) (validation / train-fit only; the TEST SET is never used). Everything below is fixed BEFORE
# any S5 result is seen (2026-10-21, ~09:55 Asia/Dhaka):
# PER-IMAGE SCORES - direction fixed in advance: LOWER = BETTER for every score.
#   F1 physics (score-FPE): H = mean over t in {10, 50} of r1 r2 / S_ref(t)^2 (s5_score.py)
#   F2 physics (film/digitiser noise): Poisson-Gaussian noise-level-function residual of the image vs the NLF fitted on the
#      real train-fit images (pinn_physics.PhysicsNLF, per-image min-max to [0,1] for every image)
#   F3 non-physics (model confidence): mean over t in {50, 200, 500} of eps-MSE(t) / median over REAL validation images
#   F4 trivial: Euclidean distance of (breast brightness, saturation share, high-frequency energy, breast area), each
#      z-scored with the real validation images of the same class, to the real class mean
#   F5 REFERENCE ONLY (never a pass criterion: it uses the same feature space as KID): mean distance to the 5 nearest
#      train-fit images in mammography-classifier feature space
# TESTS
#   T1 reliability of F1: Spearman(H first scoring, H independent re-scoring) on the first 80 pool images; F1 is declared
#      UNRELIABLE if rho < 0.5 (its filter result is then reported but cannot pass)
#   T2 filter (each of F1..F4): keep the lowest-score 50 % of the 600-image pool within each class (150 / class);
#      KID-mammo (unbiased MMD^2, cubic kernel, ALL train-fit images as reference) of the kept set vs 1000 random
#      class-balanced 50 % subsets. PASS if KID_kept <= 0.90 x median(random) AND KID_kept < 2.5th percentile(random).
#   T3 realism detector (each of F1..F4): AUC = P(score of a generated image > score of a real validation image), 1000
#      bootstrap CI. PASS if AUC >= 0.65 and the CI lower bound > 0.5 (direction fixed: generated = less consistent).
#   E2 per-t version of T3 for F1 (where in t generated images are least FP-consistent)
#   E3 descriptive: KID-mammo of checkpoints 35 / 40 / 41 / 42 / 44 and of the uniform weight average of 40/41/42/44
#      (same noise bank, 100 / class, paired bootstrap): how much of the apparent quality variability is attributable to
#      checkpoint selection / weight averaging. No pass rule (not assumed to improve anything).
#   E4 Epsilon Scaling at sampling (s = 1.005 / 1.01 / 1.02, epoch 40) vs s = 1: "helps" if KID <= 0.90 x and the paired
#      bootstrap CI of the difference lies below 0.
import numpy as np
from scipy.spatial import cKDTree
from scipy.stats import rankdata

kern = lambda A_, B_: (A_ @ B_.T / A_.shape[1] + 1.0) ** 3

def auc(pos, neg):
    """Mann-Whitney AUC = P(pos > neg) + 0.5 P(tie), average ranks for ties"""
    r = rankdata(np.concatenate([pos, neg]))
    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))

def auc_ci(pos, neg, n=1000, seed=0):
    rg = np.random.default_rng(seed)
    v = [auc(pos[rg.integers(0, len(pos), len(pos))], neg[rg.integers(0, len(neg), len(neg))]) for _ in range(n)]
    return [float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))]

def spearman(x, y):
    rx = np.argsort(np.argsort(x)); ry = np.argsort(np.argsort(y))
    return float(np.corrcoef(rx, ry)[0, 1])

def analyze(D, n_random=1000, n_boot=1000):
    """D: dict of numpy arrays (see s5_analyze.py). Returns (report lines, results dict)."""
    L = []; P_ = lambda *m: L.append(" ".join(str(x) for x in m)); rep = {}
    FR = D["feat_ref"]
    KRR = kern(FR, FR); kRR = (KRR.sum() - np.trace(KRR)) / (len(FR) * (len(FR) - 1)); del KRR
    def parts(G):
        return kern(G, G), kern(G, FR).mean(1)
    def mmd(KGG, h, c):
        num = c @ KGG @ c - (c * c * np.diag(KGG)).sum(); den = c.sum() ** 2 - (c * c).sum()
        return float(num / den - 2 * (c * h).sum() / c.sum() + kRR)
    P_("=" * 110); P_("S5 REPORT - physics-consistency scoring of synthetic mammograms + failure investigation (validation / train-fit only)"); P_("=" * 110)
    if "pool_y" in D and "score_pool_H" in D and "score_real_H" in D:
        yp, yv = D["pool_y"], D["real_y"]; Gp = D["feat_pool"]; ip, iv = D["istats_pool"], D["istats_real"]
        S, SR = {}, {}
        S["F1_fp"], SR["F1_fp"] = D["score_pool_H"], D["score_real_H"]
        S["F2_noise_physics"], SR["F2_noise_physics"] = D["nlf_pool"], D["nlf_real"]
        med = np.median(D["score_real_MSE"], 0)
        S["F3_denoising_error"], SR["F3_denoising_error"] = (D["score_pool_MSE"] / med).mean(1), (D["score_real_MSE"] / med).mean(1)
        def f4(st_, y):
            out = np.zeros(len(y))
            for c in (0, 1):
                mu_, sd_ = iv[yv == c].mean(0), iv[yv == c].std(0) + 1e-9
                out[y == c] = np.sqrt((((st_[y == c] - mu_) / sd_) ** 2).sum(1))
            return out
        S["F4_trivial_intensity"], SR["F4_trivial_intensity"] = f4(ip, yp), f4(iv, yv)
        S["F5_reference_knn"] = cKDTree(FR).query(Gp, k=min(5, len(FR)))[0].reshape(len(Gp), -1).mean(1)
        # ---- T1 ----
        P_("\n[T1] RELIABILITY OF THE FP SCORE (rule: Spearman >= 0.5 between two independent scorings)")
        if "score_retest_H" in D:
            k = len(D["score_retest_H"]); rel = spearman(D["score_pool_H"][:k], D["score_retest_H"])
            per_t = [spearman(D["score_pool_P"][:k, j], D["score_retest_P"][:, j]) for j in range(D["score_retest_P"].shape[1])]
            rep["T1"] = {"n": k, "spearman_H": rel, "spearman_per_t": per_t, "reliable": bool(rel >= 0.5)}
            P_(f"  {k} images: Spearman H {rel:.3f} | per t " + " ".join(f"{int(t)}:{r:.3f}" for t, r in zip(D["score_t"], per_t))
               + f" -> F1 {'RELIABLE' if rel >= 0.5 else 'UNRELIABLE'}")
        else:
            rep["T1"] = {"reliable": False}; P_("  retest missing -> F1 treated as UNRELIABLE")
        f1_ok = bool(rep["T1"]["reliable"])
        # ---- T2 ----
        KGG, hh = parts(Gp)
        rng = np.random.default_rng(0)
        idx_c = {c: np.where(yp == c)[0] for c in (0, 1)}
        keep_n = {c: len(idx_c[c]) // 2 for c in (0, 1)}
        rand = []
        for _ in range(n_random):
            cvec = np.zeros(len(yp))
            for c in (0, 1):
                cvec[rng.choice(idx_c[c], keep_n[c], replace=False)] = 1
            rand.append(mmd(KGG, hh, cvec))
        rand = np.array(rand); med_r, q025 = float(np.median(rand)), float(np.quantile(rand, 0.025))
        full = mmd(KGG, hh, np.ones(len(yp)))
        P_(f"\n[T2] FILTERS: keep the lowest-score 50 % per class ({keep_n[0]}+{keep_n[1]} of {len(yp)}); KID-mammo vs train-fit")
        P_(f"  whole pool {full:.4f} | random 50 % subsets: median {med_r:.4f}, 2.5th pct {q025:.4f}, 97.5th pct {np.quantile(rand, .975):.4f}")
        rep["T2"] = {"pool_kid": full, "random_median": med_r, "random_q025": q025, "filters": {}}
        for f, sc in S.items():
            cvec = np.zeros(len(yp))
            for c in (0, 1):
                ii = idx_c[c]; cvec[ii[np.argsort(sc[ii], kind="mergesort")[:keep_n[c]]]] = 1
            kk = mmd(KGG, hh, cvec); pct = float((rand < kk).mean() * 100); kept = cvec.astype(bool)
            ref_only = f.startswith("F5")
            passed = (not ref_only) and kk <= 0.90 * med_r and kk < q025 and (f1_ok or not f.startswith("F1"))
            rep["T2"]["filters"][f] = {"kid_kept": kk, "ratio_to_random_median": kk / med_r, "percentile_among_random": pct,
                                       "saturation_kept": float(ip[kept, 1].mean()), "hf_energy_kept": float(ip[kept, 2].mean()),
                                       "passed": bool(passed), "reference_only": ref_only}
            P_(f"  {f:22s}: KID {kk:.4f} = x{kk / med_r:.3f} of random median | percentile among random {pct:5.1f} | kept saturation "
               f"{ip[kept, 1].mean():.4f} HF {ip[kept, 2].mean():.4f} -> " + ("REFERENCE ONLY" if ref_only else ("PASS" if passed else "fail"))
               + ("" if (f1_ok or not f.startswith("F1")) else " (F1 unreliable)"))
        P_(f"  all pool: saturation {ip[:, 1].mean():.4f} HF {ip[:, 2].mean():.4f} | real validation: saturation {iv[:, 1].mean():.4f} HF {iv[:, 2].mean():.4f}")
        # ---- T3 / E2 ----
        P_("\n[T3] REAL vs GENERATED (AUC = P(generated scores worse than real); rule AUC >= 0.65 with CI lower > 0.5)")
        P_("  caveat: the real validation images were part of the diffusion model's training set (it was trained on all clean train")
        P_("  images), so a difference can also mean 'seen in training' - the filter test T2 does not have this problem")
        rep["T3"] = {}
        for f in ("F1_fp", "F2_noise_physics", "F3_denoising_error", "F4_trivial_intensity"):
            A0 = auc(S[f], SR[f]); ci = auc_ci(S[f], SR[f], n_boot)
            passed = A0 >= 0.65 and ci[0] > 0.5 and (f1_ok or not f.startswith("F1"))
            rep["T3"][f] = {"auc": A0, "ci95": ci, "passed": bool(passed), "median_generated": float(np.median(S[f])),
                            "median_real": float(np.median(SR[f]))}
            P_(f"  {f:22s}: AUC {A0:.3f} CI {ci[0]:.3f}-{ci[1]:.3f} | median generated {np.median(S[f]):.4g} vs real {np.median(SR[f]):.4g} -> "
               + ("PASS" if passed else "fail"))
        P_("\n[E2] FP CONSISTENCY PER t (generated vs real)")
        rep["E2"] = {}
        for j, t in enumerate(D["score_t"]):
            pg, pr = D["score_pool_P"][:, j], D["score_real_P"][:, j]
            A0 = auc(pg, pr); ci = auc_ci(pg, pr, n_boot)
            rep["E2"][str(int(t))] = {"auc": A0, "ci95": ci, "mean_generated": float(pg.mean()), "mean_real": float(pr.mean())}
            P_(f"  t={int(t):4d}: mean P generated {pg.mean():.4f} vs real {pr.mean():.4f} | AUC {A0:.3f} CI {ci[0]:.3f}-{ci[1]:.3f}")
        P_("\n[CORR] Spearman correlation between the scores on the generated pool")
        names = list(S); M = np.array([[spearman(S[p], S[q]) for q in names] for p in names])
        rep["score_correlations"] = {"names": names, "matrix": M.tolist()}
        P_("         " + " ".join(f"{n[:6]:>8s}" for n in names))
        for p, row in zip(names, M):
            P_(f"  {p[:6]:6s} " + " ".join(f"{v:+8.2f}" for v in row))
    else:
        P_("\n!! pool or scores missing -> filter tests skipped")
    # ---- E3 / E4 ----
    sets = sorted(k[5:] for k in D if k.startswith("feat_e3_") or k.startswith("feat_e4_"))
    if sets:
        n = min(len(D["feat_" + k]) for k in sets)
        FE = {k: parts(D["feat_" + k][:n]) for k in sets}
        C = np.random.default_rng(1).multinomial(n, np.ones(n) / n, size=n_boot).astype(float)
        ones = np.ones(n)
        kid = {k: mmd(*FE[k], ones) for k in FE}
        boot = {k: np.array([mmd(*FE[k], c) for c in C]) for k in FE}
        e3 = sorted(k for k in kid if k.startswith("e3_"))
        P_(f"\n[E3] CHECKPOINT SELECTION / WEIGHT AVERAGING (KID-mammo, {n} images, same noise; bootstrap 95 % CI; descriptive)")
        rep["E3"] = {}
        for k in e3:
            ci = [float(np.quantile(boot[k], .025)), float(np.quantile(boot[k], .975))]
            rep["E3"][k] = {"kid": kid[k], "ci95": ci}
            P_(f"  {k:12s}: KID {kid[k]:.4f} (CI {ci[0]:.4f}-{ci[1]:.4f})")
        singles = [k for k in e3 if k != "e3_avg"]
        if len(singles) > 1:
            vals = np.array([kid[k] for k in singles]); within = float(np.mean([boot[k].std() for k in singles]))
            rep["E3"]["summary"] = {"between_checkpoint_sd": float(vals.std(ddof=1)), "mean_within_bootstrap_sd": within,
                                    "range": [float(vals.min()), float(vals.max())]}
            P_(f"  single checkpoints: KID range {vals.min():.4f}-{vals.max():.4f} | between-checkpoint SD {vals.std(ddof=1):.4f} vs "
               f"sampling (bootstrap) SD {within:.4f} -> ratio {vals.std(ddof=1) / (within + 1e-12):.2f}")
            if "e3_avg" in kid:
                med_k = sorted(singles, key=lambda k: kid[k])[len(singles) // 2]
                d = boot["e3_avg"] - boot[med_k]
                rep["E3"]["avg_vs_median_checkpoint"] = {"median_checkpoint": med_k,
                                                         "diff_ci95": [float(np.quantile(d, .025)), float(np.quantile(d, .975))]}
                P_(f"  weight average vs the median single checkpoint ({med_k}): diff CI {np.quantile(d, .025):+.4f}..{np.quantile(d, .975):+.4f}")
        if "e3_ep040" in kid:
            P_("\n[E4] EPSILON SCALING at sampling (epoch 40) vs s = 1 (rule: KID <= 0.90x with paired CI < 0)")
            rep["E4"] = {}
            for k in sorted(x for x in kid if x.startswith("e4_")):
                d = boot[k] - boot["e3_ep040"]; ci = [float(np.quantile(d, .025)), float(np.quantile(d, .975))]
                ok = kid[k] <= 0.90 * kid["e3_ep040"] and ci[1] < 0
                rep["E4"][k] = {"kid": kid[k], "ratio": kid[k] / kid["e3_ep040"], "diff_ci95": ci, "helps": bool(ok)}
                P_(f"  {k:12s}: KID {kid[k]:.4f} = x{kid[k] / kid['e3_ep040']:.3f} (diff CI {ci[0]:+.4f}..{ci[1]:+.4f}) -> {'HELPS' if ok else 'no'}")
    P_("\n" + "=" * 110)
    P_("Send S5_report.txt (or S5_results.zip) and the grid_*.png images back.")
    return L, rep
''',
         "s5_analyze.py": r'''# s5_analyze.py - S5 analysis (validation / train-fit only; the TEST SET is never used). Everything below is fixed BEFORE
# any S5 result is seen (2026-10-21, ~09:55 Asia/Dhaka):
# PER-IMAGE SCORES - direction fixed in advance: LOWER = BETTER for every score.
#   F1 physics (score-FPE): H = mean over t in {10, 50} of r1 r2 / S_ref(t)^2 (s5_score.py)
#   F2 physics (film/digitiser noise): Poisson-Gaussian noise-level-function residual of the image vs the NLF fitted on the
#      real train-fit images (pinn_physics.PhysicsNLF, per-image min-max to [0,1] for every image)
#   F3 non-physics (model confidence): mean over t in {50, 200, 500} of eps-MSE(t) / median over REAL validation images
#   F4 trivial: Euclidean distance of (breast brightness, saturation share, high-frequency energy, breast area), each
#      z-scored with the real validation images of the same class, to the real class mean
#   F5 REFERENCE ONLY (never a pass criterion: it uses the same feature space as KID): mean distance to the 5 nearest
#      train-fit images in mammography-classifier feature space
# TESTS
#   T1 reliability of F1: Spearman(H first scoring, H independent re-scoring) on the first 80 pool images; F1 is declared
#      UNRELIABLE if rho < 0.5 (its filter result is then reported but cannot pass)
#   T2 filter (each of F1..F4): keep the lowest-score 50 % of the 600-image pool within each class (150 / class);
#      KID-mammo (unbiased MMD^2, cubic kernel, ALL train-fit images as reference) of the kept set vs 1000 random
#      class-balanced 50 % subsets. PASS if KID_kept <= 0.90 x median(random) AND KID_kept < 2.5th percentile(random).
#   T3 realism detector (each of F1..F4): AUC = P(score of a generated image > score of a real validation image), 1000
#      bootstrap CI. PASS if AUC >= 0.65 and the CI lower bound > 0.5 (direction fixed: generated = less consistent).
#   E2 per-t version of T3 for F1 (where in t generated images are least FP-consistent)
#   E3 descriptive: KID-mammo of checkpoints 35 / 40 / 41 / 42 / 44 and of the uniform weight average of 40/41/42/44
#      (same noise bank, 100 / class, paired bootstrap): how much of the apparent quality variability is attributable to
#      checkpoint selection / weight averaging. No pass rule (not assumed to improve anything).
#   E4 Epsilon Scaling at sampling (s = 1.005 / 1.01 / 1.02, epoch 40) vs s = 1: "helps" if KID <= 0.90 x and the paired
#      bootstrap CI of the difference lies below 0.
import os, sys, glob, json, argparse
import numpy as np, torch, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from phaseA_common import log, save_json, load_clean_meta, val_mask, find, OUT
from a6_classifier import build, prep
from pinn_physics import PhysicsNLF
from s5_stats import analyze

ap = argparse.ArgumentParser()
ap.add_argument("--n_random", type=int, default=1000)
ap.add_argument("--n_boot", type=int, default=1000)
a = ap.parse_args()
TAG = "S5-analyze"
dev = torch.device("cuda")
xtr, ytr, pat, _ = load_clean_meta("train"); vm = val_mask(pat)
XV, YV, XF = xtr[vm], ytr[vm], xtr[~vm]
J = lambda f: dict(np.load(os.path.join(OUT, f))) if os.path.exists(os.path.join(OUT, f)) else None

mam = build(); mam.load_state_dict({k: v.float() for k, v in torch.load(find("a6_clf_s0.pt")[0], map_location="cpu").items()})
mam = mam.to(dev).eval()
@torch.no_grad()
def feats(X_u8, bs=64):
    out = []
    for i in range(0, len(X_u8), bs):
        xb = torch.from_numpy(X_u8[i:i + bs]).to(dev).float().div(255).unsqueeze(1)
        with torch.autocast("cuda", dtype=torch.float16):
            h = mam.classifier[1](mam.classifier[0](mam.avgpool(mam.features(prep(xb)))))
        out.append(h.float().cpu())
    return torch.cat(out).numpy().astype(np.float64)

def istats_img(X_u8):
    """per-image version of the a9 istats: breast brightness, saturation share, HF energy, breast area"""
    I = torch.from_numpy(X_u8).float().div(255).unsqueeze(1)
    mn = I.amin(dim=(1, 2, 3), keepdim=True); mx = I.amax(dim=(1, 2, 3), keepdim=True); I = (I - mn) / (mx - mn + 1e-8)
    mu = F.avg_pool2d(I, 7, 1, 3, count_include_pad=False); m = (mu > 0.04).float(); n = m.sum((1, 2, 3)).clamp_min(1)
    hp = I - F.avg_pool2d(I, 7, 1, 3, count_include_pad=False)
    return np.stack([((I * m).sum((1, 2, 3)) / n).numpy(), ((((I > 0.95).float() * m).sum((1, 2, 3))) / n).numpy(),
                     ((hp * hp * m).sum((1, 2, 3)) / n).numpy(), m.mean((1, 2, 3)).numpy()], 1)

def mm_u8(X):
    Xf = torch.from_numpy(X).float(); mn = Xf.amin(dim=(1, 2), keepdim=True); mx = Xf.amax(dim=(1, 2), keepdim=True)
    return ((Xf - mn) / (mx - mn + 1e-8) * 255).round().clamp(0, 255).to(torch.uint8)

D = {"feat_ref": feats(XF), "real_y": YV, "istats_real": istats_img(XV)}
log(TAG, f"reference features: {D['feat_ref'].shape}")
pool, sp, sr_, st = J("gen_pool_ep040.npz"), J("score_pool.npz"), J("score_real.npz"), J("score_retest.npz")
if pool is not None:
    D.update({"pool_y": pool["y"], "feat_pool": feats(pool["x"]), "istats_pool": istats_img(pool["x"])})
    nlf = PhysicsNLF(dev); fitrep = nlf.fit(mm_u8(XF))
    log(TAG, f"noise-level function fitted on train-fit: b {fitrep['b']:.3e} a {fitrep['a']:.3e} c {fitrep['c']:.3e} | R2 {fitrep['R2_var']:.3f}")
    def nlf_res(X):
        U = mm_u8(X); out = []
        with torch.no_grad():
            for i in range(0, len(U), 32):
                out.append(nlf.residual(U[i:i + 32].to(dev).float().div(255).unsqueeze(1)).cpu().numpy())
        return np.concatenate(out).astype(np.float64)
    D["nlf_pool"], D["nlf_real"] = nlf_res(pool["x"]), nlf_res(XV)
for nm, s_ in (("pool", sp), ("real", sr_), ("retest", st)):
    if s_ is not None:
        D[f"score_{nm}_H"], D[f"score_{nm}_P"], D[f"score_{nm}_MSE"] = s_["H"], s_["P"], s_["MSE"]
        D["score_t"] = s_["t"]
for f in sorted(glob.glob(os.path.join(OUT, "gen_e[34]_*.npz"))):
    k = os.path.basename(f)[4:-4]; d = np.load(f)
    D["feat_" + k] = feats(d["x"]); D["istats_" + k] = istats_img(d["x"])
np.savez(os.path.join(OUT, "S5_data.npz"), **D)                 # everything needed to re-run s5_stats.py on a CPU
log(TAG, f"S5_data.npz written ({len(D)} arrays)")
L, rep = analyze(D, a.n_random, a.n_boot)
txt = "\n".join(L)
open(os.path.join(OUT, "S5_report.txt"), "w").write(txt)
save_json(rep, "S5_results.json")
print(txt)
'''}
for n, s in FILES.items():
    Path(n).write_text(s)
Path("/tmp/prepare_clean.py").write_text(r'''# prepare_clean.py - CLEAN patient-level CBIS-DDSM cache (rule B, decided 19 Oct 2026 after the data audit).
#   unit            = one unique image (pixel-identical after the 256x256 resize = the same image), not one CSV row
#   resolution      = PatientName in dicom_info (as before); if missing: series UID of the CSV path -> dicom_info, or
#                     the jpeg folder named by that UID holding exactly one image (recovers the 282 skipped test rows)
#   image check     = native min side >= 1000 px and not a binary mask (a full mammogram), else rejected + reported
#   label conflicts = an image carrying both BENIGN* and MALIGNANT rows -> excluded
#   split (rule B)  = patients that occur in both official train and test CSVs go to TEST (all their images);
#                     an image that occurs in both splits therefore ends up in test; train = everything else
#   no class subsampling (classes are already near-balanced)
# Writes /tmp/train_cache.npz and /tmp/test_cache.npz (x uint8 (N,256,256), y int64) exactly like the old prepare.py,
# plus /kaggle/working/clean_data/ (compressed caches with md5 + patient, manifests, summary) so that EVERY later
# notebook can attach this output and use byte-identical data.
import os, glob, json, shutil, subprocess, hashlib, collections, time
import numpy as np, pandas as pd
from PIL import Image

OUT = "/kaggle/working/clean_data"
os.makedirs(OUT, exist_ok=True)
EXPECTED = {"train": 2336, "test": 663}          # data audit, rule B (before the full-mammogram check)

def to_tmp(src_dir):
    for s in ("train", "test"):
        d = np.load(os.path.join(src_dir, f"{s}_cache.npz"))
        np.savez(f"/tmp/{s}_cache.npz", x=d["x"], y=d["y"])
        print(f"{s}: {d['x'].shape} benign={int((d['y'] == 0).sum())} malignant={int((d['y'] == 1).sum())}")

# ---------- 0. already built in this session, or attached from an earlier session -> reuse (identical bytes) ----------
if os.path.exists(os.path.join(OUT, "clean_summary.json")):
    to_tmp(OUT); print("clean caches already built in this session"); raise SystemExit(0)
prev = sorted(glob.glob("/kaggle/input/**/clean_data/clean_summary.json", recursive=True))
if prev:
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(OUT, f))
    to_tmp(OUT)
    print("clean caches REUSED from", src); raise SystemExit(0)

t0 = time.time()
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
JPEG = os.path.join(RAW, "jpeg")
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
jp = lambda rel: os.path.join(JPEG, str(rel).replace("CBIS-DDSM/jpeg/", ""))
full = info[info["SeriesDescription"] == "full mammogram images"]
by_name = full.set_index("PatientName")["image_path"].to_dict()
by_uid = full.set_index("SeriesInstanceUID")["image_path"].to_dict()
jpeg_dirs = set(os.listdir(JPEG))

# ---------- 1. every CSV row -> image file ----------
rows, how_n = [], collections.Counter()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test", ["csv/calc_case_description_test_set.csv", "csv/mass_case_description_test_set.csv"])]:
    for c in csvs:
        for _, r in pd.read_csv(os.path.join(RAW, c)).iterrows():
            parts = str(r["image file path"]).strip().split("/")
            rel, how = by_name.get(parts[0]), "patientname"
            if rel is not None and not os.path.exists(jp(rel)):
                rel = None
            if rel is None:
                for comp in parts[1:3]:
                    if comp in by_uid and os.path.exists(jp(by_uid[comp])):
                        rel, how = by_uid[comp], "series_uid_dicom_info"
                        break
            if rel is None:
                for comp in parts[1:3]:
                    if comp in jpeg_dirs:
                        fs = [f for f in os.listdir(os.path.join(JPEG, comp)) if f.lower().endswith(".jpg")]
                        if len(fs) == 1:
                            rel, how = os.path.join(comp, fs[0]), "uid_folder_single_jpeg"
                            break
            if rel is None:
                how = "unresolved"
            how_n[f"{split}:{how}"] += 1
            rows.append({"split": split, "csv": os.path.basename(c), "patient": str(r["patient_id"]),
                         "label": 1 if r["pathology"] == "MALIGNANT" else 0, "rel": rel, "how": how})
R = pd.DataFrame(rows)
print("row resolution:", dict(how_n), flush=True)
Rk = R[R.rel.notna()].copy()

# ---------- 2. load every unique file once: full-mammogram check + 256x256 array + md5 ----------
arr, md5, native, reject = {}, {}, {}, {}
files = sorted(Rk.rel.unique())
for i, rel in enumerate(files):
    with Image.open(jp(rel)) as im:
        W, H = im.size
        g = im.convert("L")
        native[rel] = (W, H)
        if min(W, H) < 1000:
            reject[rel] = f"small ({W}x{H}) - not a full mammogram"
            continue
        small = np.asarray(g.resize((512, 512)))
        if np.mean((small > 30) & (small < 225)) < 0.02:
            reject[rel] = "binary image - looks like an ROI mask"
            continue
        a = np.array(g.resize((256, 256)), dtype=np.uint8)            # identical resize to the old prepare.py
    arr[rel] = a
    md5[rel] = hashlib.md5(a.tobytes()).hexdigest()
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(files)} files ({(time.time() - t0) / 60:.1f} min)", flush=True)
rej_rows = Rk[Rk.rel.isin(reject)]
Rk = Rk[~Rk.rel.isin(reject)].copy()
Rk["md5"] = Rk.rel.map(md5)

# ---------- 3. one record per image, conflicts, rule-B split ----------
ptr, pte = set(R[(R.split == "train") & R.rel.notna()].patient), set(R[(R.split == "test") & R.rel.notna()].patient)
both = ptr & pte                                    # same definition as the data audit
img = Rk.groupby("md5").agg(rel=("rel", "first"), files=("rel", lambda s: sorted(set(s))),
                            splits=("split", lambda s: sorted(set(s))), patients=("patient", lambda s: sorted(set(s))),
                            labels=("label", lambda s: sorted(set(s))), csvs=("csv", lambda s: sorted(set(s)))).reset_index()
conflict = img.labels.map(len) > 1
clean = img[~conflict].copy()
clean["label"] = clean.labels.map(lambda s: s[0])
touch = clean.patients.map(lambda ps: any(p in both for p in ps))
in_tr = clean.splits.map(lambda s: "train" in s)
in_te = clean.splits.map(lambda s: "test" in s)
clean["final_split"] = np.where(in_te | touch, "test", "train")
clean = clean.sort_values("md5").reset_index(drop=True)
assert not (set(p for ps in clean[clean.final_split == "train"].patients for p in ps) &
            set(p for ps in clean[clean.final_split == "test"].patients for p in ps)), "patient overlap remains!"

summary = {"rule": "B: patients in both official splits -> test; label-conflict images excluded; one record per "
                   "pixel-identical image; full-mammogram check", "decided": "2026-10-19",
           "row_resolution": dict(how_n), "rejected_files": len(reject), "rejected_examples": dict(list(reject.items())[:10]),
           "rejected_rows": int(len(rej_rows)), "unique_images": int(len(img)), "label_conflict_images_excluded": int(conflict.sum()),
           "patients_in_both_official_splits": len(both), "build_minutes": None}
for s in ("train", "test"):
    d = clean[clean.final_split == s]
    x = np.stack([arr[r] for r in d.rel]); y = d.label.to_numpy().astype(np.int64)
    np.savez(f"/tmp/{s}_cache.npz", x=x, y=y)
    np.savez_compressed(os.path.join(OUT, f"{s}_cache.npz"), x=x, y=y, md5=d.md5.to_numpy().astype(str),
                        patient=d.patients.map(lambda p: "|".join(p)).to_numpy().astype(str))
    m = d[["md5", "label", "patients", "files", "csvs", "splits"]].copy()
    m["native_WxH"] = d.rel.map(lambda r: "x".join(map(str, native[r])))
    m.to_csv(os.path.join(OUT, f"manifest_{s}.csv"), index=False)
    pats = set(p for ps in d.patients for p in ps)
    summary[s] = {"images": int(len(d)), "benign": int((y == 0).sum()), "malignant": int((y == 1).sum()),
                  "patients": len(pats), "expected_from_audit": EXPECTED[s], "difference": int(len(d)) - EXPECTED[s]}
    print(f"{s}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} patients={len(pats)} "
          f"(audit expected {EXPECTED[s]}, difference {int(len(d)) - EXPECTED[s]})", flush=True)
summary["build_minutes"] = (time.time() - t0) / 60
json.dump(summary, open(os.path.join(OUT, "clean_summary.json"), "w"), indent=1, default=str)
print(f"rejected files (full-mammogram check): {len(reject)}", dict(list(reject.items())[:5]))
print("clean caches written to /tmp and", OUT, f"({summary['build_minutes']:.1f} min)")
''')
print("wrote", ", ".join(FILES))
subprocess.run([sys.executable, "/tmp/prepare_clean.py"], check=True)      # reuses clean_data/ from the input (identical bytes)

def find_in(name):
    h = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    print(f"{name}: {h[0] if h else '-- not found'}")
    return h[0] if h else None
def need(name, hint):
    p = find_in(name)
    if p is None:
        raise SystemExit(f"!! {name} not found -> {hint}")
    return p
OUT = os.environ["PA_OUT"]
BASE40 = need("base_clean_s42_w_ep040.pt", "Add Input -> clean-data-baseline-retraining")
need("A8_fp_clean.json", "Add Input -> S2 baseline audit"); need("a6_clf_s0.pt", "Add Input -> S2 baseline audit")
CK41 = need("s3_m0_w_ep041.pt", "Add Input -> the S3 notebook output")
CK42 = need("s3_m0_w_ep042.pt", "Add Input -> the S3 notebook output")
CK44 = need("s3_m0_w_ep044.pt", "Add Input -> the S4a notebook output")
CK35 = find_in("base_clean_s42_w_ep035.pt")
REUSE = {}
for ep, fn in ((42, "gen_s3_m0_ep042_gs2.npz"), (44, "gen_s3_m0_ep044_gs2.npz")):
    p = find_in(fn)
    if p:
        shutil.copy(p, f"{OUT}/gen_e3_ep{ep:03d}.npz"); REUSE[ep] = p
print("reused epoch-42/44 sample sets (same sampler and noise bank):", list(REUSE))
r = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True)
NGPU = int(r.stdout.split()[-1]) if r.returncode == 0 else 0
print("GPUs visible:", NGPU, "" if NGPU >= 2 else "!! expected 2 (Settings -> Accelerator -> GPU T4 x2)")
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True)
RC = {}

def run_one(name, gpu, argv, est_h, out_dir=None):
    """one job in a subprocess; skipped (rc = 'skipped') if it would pass BUDGET_H (0.4 h kept for the analysis)"""
    if used_h() + est_h > BUDGET_H - 0.4:
        print(f"--- {name} SKIPPED: {used_h():.2f} h used + {est_h} h estimate would pass the budget", flush=True)
        RC[name] = "skipped"; return
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
    if out_dir:
        env["PA_OUT"] = out_dir
    print(f">>> {name} started on GPU {env['CUDA_VISIBLE_DEVICES']} | session {used_h():.2f} h", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush()
            print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]} | session {used_h():.2f} h" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)

def run_queues(queues):
    """queues = {gpu: [(name, argv, est_h[, out_dir]), ...]} -> each GPU runs its list in order, the GPUs in parallel"""
    if NGPU < 2:
        queues = {0: [j for q in queues.values() for j in q]}
    ths = [threading.Thread(target=lambda g=g, q=q: [run_one(n, g, a, e, *rest) for n, a, e, *rest in q]) for g, q in queues.items()]
    for t in ths: t.start()
    for t in ths: t.join()

gen = lambda name, ckpts, n, extra=(): ["s5_generate.py", "--name", name, "--ckpts", ",".join(ckpts), "--n_per_class", str(n)] + list(extra)
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — SMOKE TEST (tiny versions of every script, ~10 min) =====================
SM = "/kaggle/working/smoke"
done_before = os.path.exists(f"{OUT}/S5_report.txt")
if RUN_SMOKE and not done_before:
    os.makedirs(SM, exist_ok=True)
    run_queues({0: [("smoke_gen_pool", gen("pool_ep040", [BASE40], 2, ["--steps", "5"]), 0.1, SM),
                    ("smoke_score_pool", ["s5_score.py", "--set", "pool", "--gen", f"{SM}/gen_pool_ep040.npz", "--ckpt", BASE40, "--limit", "4"], 0.1, SM),
                    ("smoke_score_retest", ["s5_score.py", "--set", "retest", "--gen", f"{SM}/gen_pool_ep040.npz", "--ckpt", BASE40,
                                            "--n_retest", "2"], 0.1, SM)],
                1: [("smoke_gen_avg", gen("e3_avg", [BASE40, CK41], 2, ["--steps", "5", "--eps_scale", "1.01"]), 0.1, SM),
                    ("smoke_score_real", ["s5_score.py", "--set", "real", "--ckpt", BASE40, "--limit", "4"], 0.1, SM)]})
    if all(RC.get(k) == 0 for k in ("smoke_gen_pool", "smoke_score_pool", "smoke_score_retest", "smoke_gen_avg", "smoke_score_real")):
        run_one("smoke_analyze", 0, ["s5_analyze.py", "--n_random", "50", "--n_boot", "50"], 0.1, SM)
    ok = RC.get("smoke_analyze") == 0 and os.path.exists(f"{SM}/S5_report.txt") and os.path.exists(f"{SM}/S5_data.npz")
    if not ok:
        raise SystemExit("SMOKE TEST FAILED -> nothing long was started. Send Claude this block + S5/logs/smoke_*.log")
    shutil.rmtree(SM, ignore_errors=True)
    print(f"SMOKE TEST PASSED | session {used_h():.2f} h")
else:
    print("smoke test skipped")

In [ ]:
# ===================== CELL 5 — RUN: two GPU queues in parallel =====================
t0 = time.time()
POOL = f"{OUT}/gen_pool_ep040.npz"
q0 = [("gen_pool", gen("pool_ep040", [BASE40], POOL_PER_CLASS), 1.4),
      ("score_pool", ["s5_score.py", "--set", "pool", "--gen", POOL, "--ckpt", BASE40], 1.3),
      (f"gen_e4_s{EPS_SCALES[2]:g}", gen(f"e4_s{EPS_SCALES[2]:g}", [BASE40], E3_PER_CLASS, ["--eps_scale", str(EPS_SCALES[2])]), 0.45),
      ("score_retest", ["s5_score.py", "--set", "retest", "--gen", POOL, "--ckpt", BASE40, "--n_retest", "80"], 0.2)]
q1 = [("gen_e3_ep040", gen("e3_ep040", [BASE40], E3_PER_CLASS), 0.45),
      ("gen_e3_ep041", gen("e3_ep041", [CK41], E3_PER_CLASS), 0.45),
      ("gen_e3_avg", gen("e3_avg", [BASE40, CK41, CK42, CK44], E3_PER_CLASS), 0.45)]
if CK35:
    q1.append(("gen_e3_ep035", gen("e3_ep035", [CK35], E3_PER_CLASS), 0.45))
for ep, ck in ((42, CK42), (44, CK44)):
    if ep not in REUSE:
        q1.append((f"gen_e3_ep{ep:03d}", gen(f"e3_ep{ep:03d}", [ck], E3_PER_CLASS), 0.45))
q1 += [(f"gen_e4_s{s:g}", gen(f"e4_s{s:g}", [BASE40], E3_PER_CLASS, ["--eps_scale", str(s)]), 0.45) for s in EPS_SCALES[:2]]
q1.append(("score_real", ["s5_score.py", "--set", "real", "--ckpt", BASE40], 0.8))
run_queues({0: q0, 1: q1})
print(f"===== queues done in {(time.time() - t0) / 3600:.2f} h | return codes {RC} | session {used_h():.2f} h =====")

In [ ]:
# ===================== CELL 6 — ANALYSIS + REPORT + zip =====================
run_one("analyze", 0, ["s5_analyze.py"], 0.3)
rp = f"{OUT}/S5_report.txt"
print(open(rp).read() if os.path.exists(rp) else "!! no report - see S5/logs/analyze.log")
json.dump(RC, open(f"{OUT}/return_codes.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/S5_results", "zip", OUT)
print("return codes:", RC, "| ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | session {used_h():.2f} h")